# RenalScan — 02: YOLOv8 Object Detection Training & Held-Out Test Evaluation

This notebook handles the training and evaluation pipeline for kidney stone detection using YOLOv8:
1. **Phase 3A: Sanity Check Run (2 Epochs)**: Validates pipeline components locally on CPU and measures training duration per epoch.
2. **Phase 3B: Full Model Training (50 Epochs on Google Colab GPU)**: Trains `yolov8n.pt` with early stopping (`patience=10`) and plots loss curves.
3. **Held-Out Test Set Assessment**: Evaluates the model explicitly on the held-out test split (`split='test'`) and outputs Precision, Recall, mAP50, and mAP50-95 to `models/test_metrics.txt`.
4. **Visual Inference Validation**: Runs detection on unseen test CT images and displays predicted bounding boxes with confidence scores.

In [ ]:
import os
import sys
from pathlib import Path
import time
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_YAML = PROJECT_ROOT / "data" / "data.yaml"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Config Exists: {DATA_YAML.exists()}")

In [ ]:
# 1. Run Sanity Check Training (2 Epochs locally or 10 Epochs)
print("=== PHASE 3A: STARTING LOCAL SANITY CHECK RUN ===")
from src.detection.train import train_yolo

# 2-epoch local sanity check run
sanity_results = train_yolo(epochs=2, imgsz=512, batch=16, patience=10, is_sanity=True)
print(f"\nSanity Run Finished in {sanity_results['elapsed_seconds']:.2f}s ({sanity_results['avg_sec_per_epoch']:.2f}s per epoch)")

### Phase 3B: Full Model Training (50 Epochs on Google Colab GPU)

To train for 50 full epochs with early stopping (`patience=10`), run the cell below on Google Colab GPU (`device='cuda:0'`). On a Colab T4 GPU, 50 epochs takes **~4-5 minutes**.

In [ ]:
# Uncomment and run on Google Colab GPU for full 50-epoch training:
# full_results = train_yolo(epochs=50, imgsz=512, batch=16, patience=10, is_sanity=False)

In [ ]:
# 2. Plot Training & Validation Loss Curves
results_csv = PROJECT_ROOT / "runs" / "detect" / "sanity_run" / "results.csv"
if not results_csv.exists():
    results_csv = PROJECT_ROOT / "runs" / "detect" / "train_run" / "results.csv"

if results_csv.exists():
    df_results = pd.read_csv(results_csv)
    df_results.columns = df_results.columns.str.strip()
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Box Loss
    axes[0].plot(df_results['epoch'], df_results['train/box_loss'], label='Train Box Loss', color='blue', marker='o')
    axes[0].plot(df_results['epoch'], df_results['val/box_loss'], label='Val Box Loss', color='orange', marker='o')
    axes[0].set_title('Bounding Box Loss', fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.6)
    
    # Class Loss
    axes[1].plot(df_results['epoch'], df_results['train/cls_loss'], label='Train Cls Loss', color='blue', marker='o')
    axes[1].plot(df_results['epoch'], df_results['val/cls_loss'], label='Val Cls Loss', color='orange', marker='o')
    axes[1].set_title('Class Loss', fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.6)
    
    # mAP Metrics
    axes[2].plot(df_results['epoch'], df_results['metrics/mAP50(B)'], label='mAP@0.50', color='green', marker='o')
    axes[2].plot(df_results['epoch'], df_results['metrics/mAP50-95(B)'], label='mAP@0.50:0.95', color='purple', marker='o')
    axes[2].set_title('Validation mAP Metrics', fontweight='bold')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('mAP Score')
    axes[2].legend()
    axes[2].grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    loss_plot_path = PROJECT_ROOT / "notebooks" / "yolo_training_loss_curves.png"
    plt.savefig(loss_plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved loss curves plot to: {loss_plot_path}")

In [ ]:
# 3. Display Held-Out Test Metrics Summary File
metrics_file = MODELS_DIR / "test_metrics.txt"
if metrics_file.exists():
    with open(metrics_file, "r", encoding="utf-8") as f:
        print(f.read())

In [ ]:
# 4. Run Test Set Visual Detections
from src.detection.predict import StoneDetector

detector = StoneDetector(model_path=MODELS_DIR / "detection_best.pt")
test_img_dir = PROJECT_ROOT / "data" / "test" / "images"
test_images = sorted(list(test_img_dir.glob("*.jpg")))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, img_p in enumerate(test_images):
    annotated_bgr, detections = detector.predict(img_p, conf_thresh=0.25)
    annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
    
    axes[idx].imshow(annotated_rgb)
    axes[idx].set_title(f"Test Sample {idx+1}\n({len(detections)} stone(s) detected)", fontsize=10)
    axes[idx].axis("off")

plt.suptitle("RenalScan YOLOv8 — Unseen Test CT Scan Detections", fontsize=14, fontweight='bold')
plt.tight_layout()
test_vis_path = PROJECT_ROOT / "notebooks" / "yolo_test_detections.png"
plt.savefig(test_vis_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved test detections visualization plot to: {test_vis_path}")